# Little_Guerra — Trabalho do Grau B (Compiladores)

Notebook gerado por `ferramentas/gerar_notebook.py`. Execute **Ambiente de execução → Executar tudo**.

Seções: 1) instalação · 2) frases da linguagem · 3) código do compilador · 4) programas de teste · 5) execução · 6) teste livre.

## 1. Instalação

In [ ]:
!pip -q install ply
import os
for d in ['guerrinha', 'ferramentas', 'testes/validos', 'testes/invalidos']:
    os.makedirs(d, exist_ok=True)

## 2. Frases da linguagem (palavras reservadas e mensagens)
Único lugar onde se trocam as palavras.

In [ ]:
%%writefile guerrinha/frases.py
# -*- coding: utf-8 -*-
"""
=====================================================================
  frases.py  —  ÚNICO ARQUIVO ONDE SE TROCAM PALAVRAS E MENSAGENS
=====================================================================

Tudo que é "texto do Little_Guerra" mora aqui. O analisador léxico monta
as expressões regulares das palavras reservadas a partir deste
dicionário, então para trocar uma palavra basta editar a string.

Regras de escrita das frases (decididas no projeto, ver docs/decisoes.md):
  * Diferencia maiúsculas de minúsculas: "Ninguém" != "ninguém".
  * Entre duas palavras da frase, aceita-se QUALQUER quantidade de
    espaço em branco (inclusive quebra de linha).
  * Ao redor de pontuação (vírgula, ?, .) o espaço é opcional.
  * A frase só casa se terminar numa fronteira de palavra
    ("...dólares" não casa com "...dólar").

Marcadores:
  # TODO-FRASE  -> ainda está com a palavra provisória em inglês.
                   Trocar pela frase escolhida e atualizar os testes
                   (buscar/substituir em testes/).
"""

# ---------------------------------------------------------------
# Identidade da linguagem
# ---------------------------------------------------------------
NOME_LINGUAGEM = "Little_Guerra"  # nome da linguagem
EXTENSAO = ".grr"                 # extensão dos programas-fonte

# ---------------------------------------------------------------
# Palavras reservadas  (TOKEN -> frase)
# ---------------------------------------------------------------
# A chave (à esquerda) é o nome do token usado na gramática
# (guerrinha/parser.py). NÃO mude as chaves, só as frases.
PALAVRAS_RESERVADAS = {
    # --- tipos (construção a) ---
    "TIPO_INT":   "radio",        # tipo inteiro
    "TIPO_FLOAT": "gaucha",       # tipo fracionário

    # --- condicional (construção b) ---
    "SE":         "Ninguém quer saber o preço do dólar",          # if
    "SENAOSE":    "Se algo deu certo, então tem que mudar",       # else if
    "SENAO":      ", nós não temos dólar",                        # else

    # --- seleção múltipla (construção c) ---
    "ESCOLHA":    "O time vai jogar motivado?",   # switch
    "CASO":       "Então vai perder motivado",    # case
    "PARE":       "Tem gato na tuba",             # break (encerramento de caso)
    "PADRAO":     "Butuca tira boi do mato",      # default (caso padrão)

    # --- extras (não obrigatórios pelo enunciado) ---
    "ENQUANTO":   "Dormindo no gelo e acordando suado",           # while
    "IMPRIME":    "Vou botar uma gravata pra te ouvir",           # print
}

# ---------------------------------------------------------------
# Mensagens (a "piada" aparece ANTES do erro técnico)
# ---------------------------------------------------------------
PIADA_ERRO_LEXICO = (
    "Eu não sei nada de tudo, eu nunca consegui falar com o cavalo, "
    "eu paro, falo com ele e ele não me responde. Futebol é meu forte."
)
PIADA_ERRO_SINTATICO = (
    "Tu comprou o cavalo pra ler ou pra correr? "
    "Porque se for pra ler está desfeito o negócio"
)
PIADA_ERRO_SEMANTICO = (          # variável não declarada, redeclaração, etc.
    "Não pode querer ir pra guerra com secador de cabelo"
)
PIADA_ERRO_TIPOS = (              # incompatibilidade de tipos na atribuição
    "Não dá pra servir farofa em restaurante com ventilador de teto"
)
PIADA_SUCESSO = (                 # programa aceito sem erros
    "Se Beleza fosse crime, eu seria inocente"
)


## 3. Código do compilador

In [ ]:
%%writefile guerrinha/__init__.py
"""Pacote do compilador. Ver frases.py para trocar palavras reservadas."""


In [ ]:
%%writefile guerrinha/erros.py
# -*- coding: utf-8 -*-
"""Formatação padronizada das mensagens de erro (sempre com a linha)."""

from . import frases

FRASES = {
    "léxico": frases.PIADA_ERRO_LEXICO,
    "sintático": frases.PIADA_ERRO_SINTATICO,
    "semântico": frases.PIADA_ERRO_SEMANTICO,
    "tipos": frases.PIADA_ERRO_TIPOS,
}


class Erro:
    """Um erro encontrado em qualquer fase. `categoria` escolhe a frase."""

    def __init__(self, fase, linha, mensagem, categoria=None):
        self.fase = fase                    # "léxico" | "sintático" | "semântico"
        self.linha = linha
        self.mensagem = mensagem
        self.categoria = categoria or fase  # "tipos" usa frase própria

    def __str__(self):
        frase = FRASES.get(self.categoria, "")
        return (f'  >> "{frase}"\n'
                f"  [{frases.NOME_LINGUAGEM}] Erro {self.fase} na linha "
                f"{self.linha}: {self.mensagem}")


In [ ]:
%%writefile guerrinha/lexer.py
# -*- coding: utf-8 -*-
"""
Analisador léxico (PLY / ply.lex).

As palavras reservadas são FRASES, então não dá para usar o truque clássico
de "reconhece ID e consulta uma tabela". Em vez disso, cada frase de
frases.PALAVRAS_RESERVADAS vira uma expressão regular, e todas são testadas
ANTES do identificador (no PLY, tokens definidos por função são tentados na
ordem em que aparecem no arquivo).
"""

import re
import ply.lex as lex
from ply.lex import TOKEN

from . import frases
from .erros import Erro


# ------------------------------------------------------------------
# Construção das regex das frases
# ------------------------------------------------------------------
def frase_para_regex(frase):
    """'Se algo deu certo, então' -> r'Se\\s+algo\\s+deu\\s+certo\\s*,\\s*então(?!\\w)'"""
    pedacos = re.findall(r"\w+|[^\w\s]", frase)
    regex = ""
    for i, p in enumerate(pedacos):
        if i > 0:
            ambos_palavra = pedacos[i - 1][-1].isalnum() and p[0].isalnum()
            regex += r"\s+" if ambos_palavra else r"\s*"
        regex += re.escape(p)
    if pedacos and pedacos[-1][-1].isalnum():
        regex += r"(?!\w)"           # fronteira de palavra no fim
    return regex


# Frases mais longas primeiro: evita que uma frase que é prefixo de outra
# "roube" o casamento.
_ORDEM = sorted(frases.PALAVRAS_RESERVADAS.items(), key=lambda kv: -len(kv[1]))
_REGEX_FRASE = {tok: re.compile(frase_para_regex(f)) for tok, f in _ORDEM}
REGEX_PALAVRAS = "|".join(f"(?:{frase_para_regex(f)})" for _, f in _ORDEM)


# ------------------------------------------------------------------
# Lista de tokens
# ------------------------------------------------------------------
tokens = tuple(frases.PALAVRAS_RESERVADAS.keys()) + (
    "ID", "NUM_INT", "NUM_FLOAT", "TEXTO",
    "MAIS", "MENOS", "VEZES", "DIVIDE",
    "IGUAL", "DIFERENTE", "MAIOR", "MENOR", "MAIOR_IGUAL", "MENOR_IGUAL",
    "ATRIBUI", "PONTO_VIRGULA", "VIRGULA", "DOIS_PONTOS",
    "ABRE_PAR", "FECHA_PAR", "ABRE_CHAVE", "FECHA_CHAVE",
)

# Símbolos simples (o PLY ordena estas strings da regex mais longa para a
# mais curta, por isso '==' é tentado antes de '=' e '>=' antes de '>').
t_MAIS = r"\+"
t_MENOS = r"-"
t_VEZES = r"\*"
t_DIVIDE = r"/"
t_IGUAL = r"=="
t_DIFERENTE = r"!="
t_MAIOR_IGUAL = r">="
t_MENOR_IGUAL = r"<="
t_MAIOR = r">"
t_MENOR = r"<"
t_ATRIBUI = r"="
t_PONTO_VIRGULA = r";"
t_VIRGULA = r","
t_DOIS_PONTOS = r":"
t_ABRE_PAR = r"\("
t_FECHA_PAR = r"\)"
t_ABRE_CHAVE = r"\{"
t_FECHA_CHAVE = r"\}"

t_ignore = " \t\r"


# ------------------------------------------------------------------
# Tokens definidos por função — A ORDEM IMPORTA
# ------------------------------------------------------------------
def t_COMENTARIO(t):
    r"//[^\n]*"
    pass  # descartado


def t_TEXTO(t):
    r'"[^"\n]*"'
    # Texto entre aspas duplas, numa linha só, sem sequências de escape.
    t.value = t.value[1:-1]
    return t


def t_TEXTO_ABERTO(t):
    r'"[^"\n]*'
    # Aspas abertas que não fecham até o fim da linha.
    t.lexer.erros.append(Erro(
        "léxico", t.lexer.lineno,
        "texto sem aspas de fechamento (o texto precisa abrir e fechar "
        "na mesma linha)."))
    return None


@TOKEN(REGEX_PALAVRAS)
def t_PALAVRA_RESERVADA(t):
    # Descobre QUAL frase casou e troca o tipo do token.
    for tok, rx in _REGEX_FRASE.items():
        if rx.fullmatch(t.value):
            t.type = tok
            break
    t.lexer.lineno += t.value.count("\n")   # frase quebrada em linhas
    return t


def t_NUM_FLOAT(t):
    r"\d+\.\d+"
    t.value = float(t.value)
    return t


def t_NUM_INT(t):
    r"\d+"
    t.value = int(t.value)
    return t


def t_ID(t):
    r"[^\W\d]\w*"
    # A regex aceita letras acentuadas só para dar UMA mensagem por palavra
    # (ex.: uma frase reservada digitada errado). Identificador válido é ASCII.
    if not t.value.isascii():
        t.lexer.erros.append(Erro(
            "léxico", t.lexer.lineno,
            f"palavra {t.value!r} não reconhecida: identificadores não podem "
            f"ter acento (será que é uma frase reservada digitada errado?)."))
        return None
    return t


def t_newline(t):
    r"\n+"
    t.lexer.lineno += len(t.value)


def t_error(t):
    t.lexer.erros.append(Erro(
        "léxico", t.lexer.lineno,
        f"caractere inesperado {t.value[0]!r}."))
    t.lexer.skip(1)


def criar_lexer():
    lexer = lex.lex(reflags=re.UNICODE)
    lexer.erros = []
    return lexer


In [ ]:
%%writefile guerrinha/ast_nodes.py
# -*- coding: utf-8 -*-
"""
Nós da árvore sintática abstrata (AST).

A AST é construída nas AÇÕES SEMÂNTICAS do parser (tradução dirigida por
sintaxe). Cada nó guarda a linha de origem para as mensagens de erro.
`tipo` é preenchido depois, pela verificação semântica.
"""

from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class No:
    linha: int


# ---------------- Expressões ----------------
@dataclass
class Num(No):
    valor: object                 # int ou float
    tipo: Optional[str] = None


@dataclass
class Var(No):
    nome: str
    tipo: Optional[str] = None


@dataclass
class BinOp(No):
    op: str                       # + - * /
    esq: No
    dir: No
    tipo: Optional[str] = None


@dataclass
class Negativo(No):               # menos unário
    expr: No
    tipo: Optional[str] = None


@dataclass
class Condicao(No):
    op: str                       # == != > < >= <=
    esq: No
    dir: No


# ---------------- Comandos ----------------
@dataclass
class DeclItem(No):
    nome: str
    init: Optional[No] = None


@dataclass
class Declaracao(No):
    tipo: str                     # "int" | "float"
    itens: List[DeclItem]


@dataclass
class Atribuicao(No):
    nome: str
    expr: No
    tipo: Optional[str] = None    # tipo da variável-alvo (preenchido na semântica)


@dataclass
class Bloco(No):
    comandos: List[No]


@dataclass
class Se(No):
    cond: Condicao
    entao: Bloco
    senaoses: List["SenaoSe"] = field(default_factory=list)
    senao: Optional[Bloco] = None


@dataclass
class SenaoSe(No):
    cond: Condicao
    bloco: Bloco


@dataclass
class Enquanto(No):
    cond: Condicao
    corpo: Bloco


@dataclass
class Caso(No):
    rotulos: List[int]            # vazio = caso padrão
    comandos: List[No]


@dataclass
class Escolha(No):
    expr: No
    casos: List[Caso]
    padrao: Optional[Caso] = None


@dataclass
class Pare(No):
    pass


@dataclass
class Texto(No):                  # literal de texto; só aparece no Imprime
    valor: str


@dataclass
class Imprime(No):
    expr: No                      # expressão numérica ou Texto


@dataclass
class Programa(No):
    comandos: List[No]


# ---------------- Impressão da AST ----------------
def imprimir_ast(no, nivel=0):
    """Devolve a AST como texto indentado."""
    pad = "  " * nivel
    linhas = []

    def filho(rotulo, n):
        linhas.append(f"{pad}  {rotulo}:")
        linhas.append(imprimir_ast(n, nivel + 2))

    if isinstance(no, Programa):
        linhas.append(f"{pad}Programa")
        for c in no.comandos:
            linhas.append(imprimir_ast(c, nivel + 1))
    elif isinstance(no, Declaracao):
        linhas.append(f"{pad}Declaração {no.tipo}  (linha {no.linha})")
        for it in no.itens:
            linhas.append(f"{pad}  {it.nome}")
            if it.init is not None:
                linhas.append(imprimir_ast(it.init, nivel + 2))
    elif isinstance(no, Atribuicao):
        linhas.append(f"{pad}Atribuição {no.nome} =  (linha {no.linha})")
        linhas.append(imprimir_ast(no.expr, nivel + 1))
    elif isinstance(no, Bloco):
        for c in no.comandos:
            linhas.append(imprimir_ast(c, nivel))
    elif isinstance(no, Se):
        linhas.append(f"{pad}Se  (linha {no.linha})")
        filho("condição", no.cond)
        filho("então", no.entao)
        for s in no.senaoses:
            filho(f"senão-se (linha {s.linha}) condição", s.cond)
            filho("bloco", s.bloco)
        if no.senao is not None:
            filho("senão", no.senao)
    elif isinstance(no, Enquanto):
        linhas.append(f"{pad}Enquanto  (linha {no.linha})")
        filho("condição", no.cond)
        filho("corpo", no.corpo)
    elif isinstance(no, Escolha):
        linhas.append(f"{pad}Escolha  (linha {no.linha})")
        filho("expressão", no.expr)
        for c in no.casos + ([no.padrao] if no.padrao else []):
            rot = ", ".join(map(str, c.rotulos)) if c.rotulos else "padrão"
            linhas.append(f"{pad}  caso {rot}:")
            for cmd in c.comandos:
                linhas.append(imprimir_ast(cmd, nivel + 2))
    elif isinstance(no, Pare):
        linhas.append(f"{pad}Pare  (linha {no.linha})")
    elif isinstance(no, Imprime):
        linhas.append(f"{pad}Imprime  (linha {no.linha})")
        linhas.append(imprimir_ast(no.expr, nivel + 1))
    elif isinstance(no, Condicao):
        linhas.append(f"{pad}{no.op}")
        linhas.append(imprimir_ast(no.esq, nivel + 1))
        linhas.append(imprimir_ast(no.dir, nivel + 1))
    elif isinstance(no, BinOp):
        sufixo = f"  : {no.tipo}" if no.tipo else ""
        linhas.append(f"{pad}{no.op}{sufixo}")
        linhas.append(imprimir_ast(no.esq, nivel + 1))
        linhas.append(imprimir_ast(no.dir, nivel + 1))
    elif isinstance(no, Negativo):
        linhas.append(f"{pad}- (unário)")
        linhas.append(imprimir_ast(no.expr, nivel + 1))
    elif isinstance(no, Texto):
        linhas.append(f'{pad}"{no.valor}"')
    elif isinstance(no, Num):
        linhas.append(f"{pad}{no.valor}")
    elif isinstance(no, Var):
        linhas.append(f"{pad}{no.nome}")
    else:
        linhas.append(f"{pad}?? {type(no).__name__}")
    return "\n".join(l for l in linhas if l != "")


In [ ]:
%%writefile guerrinha/parser.py
# -*- coding: utf-8 -*-
"""
Analisador sintático (PLY / ply.yacc, LALR(1)).

Cada função p_* é uma produção: a docstring é a regra BNF e o corpo é a
ação semântica, que constrói o nó da AST (tradução dirigida por sintaxe).
A gramática completa, em BNF, está em docs/gramatica.md.
"""

import inspect
import ply.yacc as yacc

from . import frases
from .erros import Erro
from .lexer import tokens, criar_lexer  # noqa: F401  (o PLY lê `tokens`)
from . import ast_nodes as A


class ErroSintatico(Exception):
    def __init__(self, erro):
        super().__init__(str(erro))
        self.erro = erro


# ------------------------------------------------------------------
# Precedência e associatividade (da MENOR para a MAIOR)
# Resolve os conflitos shift/reduce da gramática ambígua de expressões.
# ------------------------------------------------------------------
precedence = (
    ("left", "MAIS", "MENOS"),
    ("left", "VEZES", "DIVIDE"),
    ("right", "MENOS_UNARIO"),          # token fictício, usado com %prec
)

start = "programa"


# ------------------------------------------------------------------
# Programa e listas de comandos
# ------------------------------------------------------------------
def p_programa(p):
    "programa : lista_cmd"
    p[0] = A.Programa(1, p[1])


def p_lista_cmd(p):
    "lista_cmd : lista_cmd cmd"
    p[0] = p[1] + [p[2]]


def p_lista_cmd_vazia(p):
    "lista_cmd : "
    p[0] = []


def p_cmd(p):
    """cmd : declaracao
           | atribuicao
           | se
           | escolha
           | enquanto
           | imprime
           | pare"""
    p[0] = p[1]


def p_bloco(p):
    "bloco : ABRE_CHAVE lista_cmd FECHA_CHAVE"
    p[0] = A.Bloco(p.lineno(1), p[2])


# ------------------------------------------------------------------
# (a) Declaração de variáveis — individual ou em lista, com init opcional
# ------------------------------------------------------------------
def p_declaracao(p):
    "declaracao : tipo lista_decl PONTO_VIRGULA"
    nome_tipo, linha = p[1]
    p[0] = A.Declaracao(linha, nome_tipo, p[2])


def p_tipo(p):
    """tipo : TIPO_INT
            | TIPO_FLOAT"""
    nome = "int" if p.slice[1].type == "TIPO_INT" else "float"
    p[0] = (nome, p.lineno(1))


def p_lista_decl(p):
    "lista_decl : lista_decl VIRGULA item_decl"
    p[0] = p[1] + [p[3]]


def p_lista_decl_um(p):
    "lista_decl : item_decl"
    p[0] = [p[1]]


def p_item_decl(p):
    "item_decl : ID"
    p[0] = A.DeclItem(p.lineno(1), p[1])


def p_item_decl_init(p):
    "item_decl : ID ATRIBUI expr"
    p[0] = A.DeclItem(p.lineno(1), p[1], p[3])


# ------------------------------------------------------------------
# (e) Atribuição
# ------------------------------------------------------------------
def p_atribuicao(p):
    "atribuicao : ID ATRIBUI expr PONTO_VIRGULA"
    p[0] = A.Atribuicao(p.lineno(1), p[1], p[3])


# ------------------------------------------------------------------
# (b) Condicional: se / senão-se* / senão?
# Blocos com chaves obrigatórias => não há "dangling else".
# ------------------------------------------------------------------
def p_se(p):
    "se : SE ABRE_PAR condicao FECHA_PAR bloco lista_senaose senao_opt"
    p[0] = A.Se(p.lineno(1), p[3], p[5], p[6], p[7])


def p_lista_senaose(p):
    "lista_senaose : lista_senaose SENAOSE ABRE_PAR condicao FECHA_PAR bloco"
    p[0] = p[1] + [A.SenaoSe(p.lineno(2), p[4], p[6])]


def p_lista_senaose_vazia(p):
    "lista_senaose : "
    p[0] = []


def p_senao_opt(p):
    "senao_opt : SENAO bloco"
    p[0] = p[2]


def p_senao_opt_vazio(p):
    "senao_opt : "
    p[0] = None


# ------------------------------------------------------------------
# Laço (extra)
# ------------------------------------------------------------------
def p_enquanto(p):
    "enquanto : ENQUANTO ABRE_PAR condicao FECHA_PAR bloco"
    p[0] = A.Enquanto(p.lineno(1), p[3], p[5])


# ------------------------------------------------------------------
# (c) Seleção múltipla
# ------------------------------------------------------------------
def p_escolha(p):
    "escolha : ESCOLHA ABRE_PAR expr FECHA_PAR ABRE_CHAVE lista_casos padrao_opt FECHA_CHAVE"
    p[0] = A.Escolha(p.lineno(1), p[3], p[6], p[7])


def p_lista_casos(p):
    "lista_casos : lista_casos caso"
    p[0] = p[1] + [p[2]]


def p_lista_casos_um(p):
    "lista_casos : caso"
    p[0] = [p[1]]


def p_caso(p):
    "caso : CASO lista_rotulos DOIS_PONTOS lista_cmd"
    p[0] = A.Caso(p.lineno(1), p[2], p[4])


def p_lista_rotulos(p):
    "lista_rotulos : lista_rotulos VIRGULA rotulo"
    p[0] = p[1] + [p[3]]


def p_lista_rotulos_um(p):
    "lista_rotulos : rotulo"
    p[0] = [p[1]]


def p_rotulo(p):
    "rotulo : NUM_INT"
    p[0] = p[1]


def p_rotulo_negativo(p):
    "rotulo : MENOS NUM_INT"
    p[0] = -p[2]


def p_padrao_opt(p):
    "padrao_opt : PADRAO DOIS_PONTOS lista_cmd"
    p[0] = A.Caso(p.lineno(1), [], p[3])


def p_padrao_opt_vazio(p):
    "padrao_opt : "
    p[0] = None


def p_pare(p):
    "pare : PARE PONTO_VIRGULA"
    p[0] = A.Pare(p.lineno(1))


# ------------------------------------------------------------------
# Saída (extra)
# ------------------------------------------------------------------
def p_imprime(p):
    "imprime : IMPRIME ABRE_PAR expr FECHA_PAR PONTO_VIRGULA"
    p[0] = A.Imprime(p.lineno(1), p[3])


def p_imprime_texto(p):
    "imprime : IMPRIME ABRE_PAR TEXTO FECHA_PAR PONTO_VIRGULA"
    p[0] = A.Imprime(p.lineno(1), A.Texto(p.lineno(3), p[3]))


# ------------------------------------------------------------------
# (f) Condições com os seis operadores relacionais
# ------------------------------------------------------------------
def p_condicao(p):
    """condicao : expr IGUAL expr
                | expr DIFERENTE expr
                | expr MAIOR expr
                | expr MENOR expr
                | expr MAIOR_IGUAL expr
                | expr MENOR_IGUAL expr"""
    p[0] = A.Condicao(p.lineno(2), p[2], p[1], p[3])


# ------------------------------------------------------------------
# (d) Expressões aritméticas
# Gramática ambígua + tabela `precedence` (em vez de expr/termo/fator).
# ------------------------------------------------------------------
def p_expr_binaria(p):
    """expr : expr MAIS expr
            | expr MENOS expr
            | expr VEZES expr
            | expr DIVIDE expr"""
    p[0] = A.BinOp(p.lineno(2), p[2], p[1], p[3])


def p_expr_negativo(p):
    "expr : MENOS expr %prec MENOS_UNARIO"
    p[0] = A.Negativo(p.lineno(1), p[2])


def p_expr_parenteses(p):
    "expr : ABRE_PAR expr FECHA_PAR"
    p[0] = p[2]


def p_expr_int(p):
    "expr : NUM_INT"
    p[0] = A.Num(p.lineno(1), p[1], "int")


def p_expr_float(p):
    "expr : NUM_FLOAT"
    p[0] = A.Num(p.lineno(1), p[1], "float")


def p_expr_id(p):
    "expr : ID"
    p[0] = A.Var(p.lineno(1), p[1])


# ------------------------------------------------------------------
# Erro sintático: para no primeiro erro (sem recuperação).
# ------------------------------------------------------------------
def _descrever(tok):
    if tok.type in frases.PALAVRAS_RESERVADAS:
        return f'palavra reservada "{frases.PALAVRAS_RESERVADAS[tok.type]}"'
    return f"{tok.value!r}"


def p_error(tok):
    if tok is None:
        linha = getattr(_lexer_atual, "ultima_linha", 0)
        msg = "fim de arquivo inesperado (faltou fechar algum bloco ou ';'?)."
    else:
        linha = tok.lineno
        msg = f"token inesperado {_descrever(tok)}."
    raise ErroSintatico(Erro("sintático", linha, msg))


# ------------------------------------------------------------------
# Construção do parser
# ------------------------------------------------------------------
_lexer_atual = None


def criar_parser(debug=False, outputdir=None):
    """debug=True grava parser.out (tabela LALR e conflitos) em outputdir."""
    kwargs = dict(debug=debug)
    if outputdir:
        kwargs["outputdir"] = outputdir
    if "write_tables" in inspect.signature(yacc.yacc).parameters:
        kwargs["write_tables"] = False      # PLY 3.11: não gerar parsetab.py
    return yacc.yacc(**kwargs)


def analisar(codigo, debug=False, outputdir=None):
    """Devolve (ast, erros). Se houver erro léxico ou sintático, ast = None."""
    global _lexer_atual
    # 1ª passada: só o léxico, para relatar TODOS os erros léxicos de uma vez
    # e não gerar erros sintáticos "em cascata" por causa deles.
    lexer = criar_lexer()
    lexer.input(codigo)
    for _ in lexer:
        pass
    if lexer.erros:
        return None, lexer.erros

    # 2ª passada: o parser puxa os tokens do lexer sob demanda.
    lexer = criar_lexer()
    _lexer_atual = lexer
    lexer.ultima_linha = 1
    _token_original = lexer.token

    def token_com_linha():          # lembra a linha do último token lido
        tok = _token_original()
        if tok is not None:
            lexer.ultima_linha = tok.lineno
        return tok
    lexer.token = token_com_linha
    parser = criar_parser(debug, outputdir)
    try:
        ast = parser.parse(codigo, lexer=lexer)
    except ErroSintatico as e:
        return None, [e.erro]
    return ast, []


In [ ]:
%%writefile guerrinha/semantica.py
# -*- coding: utf-8 -*-
"""
Verificação semântica (percorre a AST).

Regras implementadas:
  S1. Variável deve ser declarada antes do uso.
  S2. Não pode redeclarar um nome visível (mesmo escopo ou escopo externo).
  S3. Atribuição: int <- float é ERRO; float <- int é permitido (alargamento).
  S4. Expressão do comando de seleção múltipla deve ser int.
  S5. Rótulos de caso não podem se repetir no mesmo comando.
  S6. Encerramento de caso (break) só dentro de seleção múltipla ou laço.

Escopo: cada bloco { } abre um escopo novo (pilha de tabelas de símbolos).
Tipos de expressão: int op int -> int (divisão inteira); qualquer float -> float.
"""

from . import ast_nodes as A
from . import frases
from .erros import Erro

# Nome de cada tipo interno como o programador escreve na linguagem
# (as mensagens de erro usam a palavra reservada, não o nome interno).
NOME_TIPO = {
    "int": frases.PALAVRAS_RESERVADAS["TIPO_INT"],
    "float": frases.PALAVRAS_RESERVADAS["TIPO_FLOAT"],
}


class TabelaSimbolos:
    def __init__(self):
        self.escopos = [{}]

    def abrir(self):
        self.escopos.append({})

    def fechar(self):
        self.escopos.pop()

    def buscar(self, nome):
        for escopo in reversed(self.escopos):
            if nome in escopo:
                return escopo[nome]
        return None

    def declarar(self, nome, tipo, linha):
        self.escopos[-1][nome] = {"tipo": tipo, "linha": linha}


class Verificador:
    def __init__(self):
        self.ts = TabelaSimbolos()
        self.erros = []
        self.profundidade_pare = 0      # >0 = dentro de escolha/enquanto

    # ---------------- utilitários ----------------
    def erro(self, linha, msg, categoria="semântico"):
        self.erros.append(Erro("semântico", linha, msg, categoria))

    def verificar(self, programa):
        for c in programa.comandos:
            self.cmd(c)
        return self.erros

    # ---------------- comandos ----------------
    def cmd(self, no):
        metodo = getattr(self, "cmd_" + type(no).__name__)
        metodo(no)

    def bloco(self, bloco):
        self.ts.abrir()
        for c in bloco.comandos:
            self.cmd(c)
        self.ts.fechar()

    def cmd_Declaracao(self, no):
        for it in no.itens:
            existente = self.ts.buscar(it.nome)
            if existente:
                self.erro(it.linha,
                          f"variável '{it.nome}' já foi declarada na linha "
                          f"{existente['linha']}.")
                continue
            if it.init is not None:
                t = self.expr(it.init)
                self.checar_atribuicao(it.nome, no.tipo, t, it.linha)
            self.ts.declarar(it.nome, no.tipo, it.linha)

    def cmd_Atribuicao(self, no):
        simbolo = self.ts.buscar(no.nome)
        t_expr = self.expr(no.expr)
        if simbolo is None:
            self.erro(no.linha, f"variável '{no.nome}' não foi declarada.")
            return
        no.tipo = simbolo["tipo"]
        self.checar_atribuicao(no.nome, simbolo["tipo"], t_expr, no.linha)

    def checar_atribuicao(self, nome, t_var, t_expr, linha):
        if t_var == "int" and t_expr == "float":
            self.erro(linha,
                      f"não é possível atribuir valor {NOME_TIPO['float']} "
                      f"à variável {NOME_TIPO['int']} '{nome}'.",
                      categoria="tipos")

    def cmd_Se(self, no):
        self.condicao(no.cond)
        self.bloco(no.entao)
        for s in no.senaoses:
            self.condicao(s.cond)
            self.bloco(s.bloco)
        if no.senao is not None:
            self.bloco(no.senao)

    def cmd_Enquanto(self, no):
        self.condicao(no.cond)
        self.profundidade_pare += 1
        self.bloco(no.corpo)
        self.profundidade_pare -= 1

    def cmd_Escolha(self, no):
        t = self.expr(no.expr)
        if t == "float":
            self.erro(no.linha, f"a expressão da seleção múltipla deve ser "
                                f"{NOME_TIPO['int']}, mas é "
                                f"{NOME_TIPO['float']}.", categoria="tipos")
        vistos = {}
        self.profundidade_pare += 1
        for caso in no.casos:
            for r in caso.rotulos:
                if r in vistos:
                    self.erro(caso.linha, f"rótulo de caso {r} repetido "
                                          f"(já usado na linha {vistos[r]}).")
                else:
                    vistos[r] = caso.linha
        # Todos os casos compartilham um escopo (como em C): há "fallthrough".
        self.ts.abrir()
        for caso in no.casos + ([no.padrao] if no.padrao else []):
            for c in caso.comandos:
                self.cmd(c)
        self.ts.fechar()
        self.profundidade_pare -= 1

    def cmd_Pare(self, no):
        if self.profundidade_pare == 0:
            self.erro(no.linha, "encerramento de caso fora de seleção "
                                "múltipla ou laço.")

    def cmd_Imprime(self, no):
        if not isinstance(no.expr, A.Texto):   # texto não tem o que verificar
            self.expr(no.expr)

    # ---------------- expressões ----------------
    def condicao(self, cond):
        self.expr(cond.esq)
        self.expr(cond.dir)

    def expr(self, no):
        """Devolve o tipo ('int' | 'float') ou None se houve erro."""
        if isinstance(no, A.Num):
            return no.tipo
        if isinstance(no, A.Var):
            s = self.ts.buscar(no.nome)
            if s is None:
                self.erro(no.linha, f"variável '{no.nome}' não foi declarada.")
                return None
            no.tipo = s["tipo"]
            return no.tipo
        if isinstance(no, A.Negativo):
            no.tipo = self.expr(no.expr)
            return no.tipo
        if isinstance(no, A.BinOp):
            te, td = self.expr(no.esq), self.expr(no.dir)
            if te is None or td is None:
                return None
            no.tipo = "float" if "float" in (te, td) else "int"
            return no.tipo
        raise TypeError(f"nó de expressão desconhecido: {no}")


def verificar(programa):
    return Verificador().verificar(programa)


In [ ]:
%%writefile guerrinha/tac.py
# -*- coding: utf-8 -*-
"""
Geração de código de três endereços (TAC) a partir da AST já verificada.

Formato das instruções:
    x = y op z          operação binária (op em + - * /)
    x = - y             menos unário
    x = y               cópia
    x = (float) y       conversão int -> float (alargamento implícito)
    ifFalse a rel b goto L
    if a == b goto L    (usado na seleção múltipla)
    goto L
    L:                  rótulo
    print x
    print "texto"
"""

from . import ast_nodes as A


class GeradorTAC:
    def __init__(self):
        self.codigo = []
        self.n_temp = 0
        self.n_rot = 0
        self.alvos_pare = []      # pilha de rótulos de saída (break)

    # ---------------- utilitários ----------------
    def temp(self):
        self.n_temp += 1
        return f"t{self.n_temp}"

    def rotulo(self):
        self.n_rot += 1
        return f"L{self.n_rot}"

    def emit(self, instr):
        self.codigo.append("    " + instr)

    def emit_rotulo(self, r):
        self.codigo.append(f"{r}:")

    def gerar(self, programa):
        for c in programa.comandos:
            self.cmd(c)
        return self.codigo

    # ---------------- expressões ----------------
    def expr(self, no):
        """Gera código para a expressão e devolve o endereço do resultado."""
        if isinstance(no, A.Num):
            return str(no.valor)
        if isinstance(no, A.Var):
            return no.nome
        if isinstance(no, A.Negativo):
            a = self.expr(no.expr)
            t = self.temp()
            self.emit(f"{t} = - {a}")
            return t
        if isinstance(no, A.BinOp):
            a = self.para_tipo(no.esq, no.tipo)
            b = self.para_tipo(no.dir, no.tipo)
            t = self.temp()
            self.emit(f"{t} = {a} {no.op} {b}")
            return t
        raise TypeError(no)

    def para_tipo(self, no, tipo_destino):
        """Gera a expressão e converte int -> float se necessário."""
        end = self.expr(no)
        if tipo_destino == "float" and getattr(no, "tipo", None) == "int":
            if isinstance(no, A.Num):
                return f"{float(no.valor)}"
            t = self.temp()
            self.emit(f"{t} = (float) {end}")
            return t
        return end

    def condicao_falsa(self, cond, destino):
        tipo = "float" if "float" in (cond.esq.tipo, cond.dir.tipo) else "int"
        a = self.para_tipo(cond.esq, tipo)
        b = self.para_tipo(cond.dir, tipo)
        self.emit(f"ifFalse {a} {cond.op} {b} goto {destino}")

    # ---------------- comandos ----------------
    def cmd(self, no):
        getattr(self, "cmd_" + type(no).__name__)(no)

    def bloco(self, b):
        for c in b.comandos:
            self.cmd(c)

    def cmd_Declaracao(self, no):
        for it in no.itens:
            if it.init is not None:
                self.emit(f"{it.nome} = {self.para_tipo(it.init, no.tipo)}")

    def cmd_Atribuicao(self, no):
        self.emit(f"{no.nome} = {self.para_tipo(no.expr, no.tipo)}")

    def cmd_Se(self, no):
        if not no.senaoses and no.senao is None:     # se simples
            fim = self.rotulo()
            self.condicao_falsa(no.cond, fim)
            self.bloco(no.entao)
            self.emit_rotulo(fim)
            return
        fim = self.rotulo()
        prox = self.rotulo()
        self.condicao_falsa(no.cond, prox)
        self.bloco(no.entao)
        self.emit(f"goto {fim}")
        self.emit_rotulo(prox)
        for s in no.senaoses:
            prox = self.rotulo()
            self.condicao_falsa(s.cond, prox)
            self.bloco(s.bloco)
            self.emit(f"goto {fim}")
            self.emit_rotulo(prox)
        if no.senao is not None:
            self.bloco(no.senao)
        self.emit_rotulo(fim)

    def cmd_Enquanto(self, no):
        inicio, fim = self.rotulo(), self.rotulo()
        self.emit_rotulo(inicio)
        self.condicao_falsa(no.cond, fim)
        self.alvos_pare.append(fim)
        self.bloco(no.corpo)
        self.alvos_pare.pop()
        self.emit(f"goto {inicio}")
        self.emit_rotulo(fim)

    def cmd_Escolha(self, no):
        t = self.expr(no.expr)
        fim = self.rotulo()
        rot_casos = [self.rotulo() for _ in no.casos]
        rot_padrao = self.rotulo() if no.padrao else fim
        # despacho
        for caso, r in zip(no.casos, rot_casos):
            for valor in caso.rotulos:
                self.emit(f"if {t} == {valor} goto {r}")
        self.emit(f"goto {rot_padrao}")
        # corpos (sem 'pare' o fluxo cai no próximo caso, como em C)
        self.alvos_pare.append(fim)
        for caso, r in zip(no.casos, rot_casos):
            self.emit_rotulo(r)
            for c in caso.comandos:
                self.cmd(c)
        if no.padrao:
            self.emit_rotulo(rot_padrao)
            for c in no.padrao.comandos:
                self.cmd(c)
        self.alvos_pare.pop()
        self.emit_rotulo(fim)

    def cmd_Pare(self, no):
        self.emit(f"goto {self.alvos_pare[-1]}")

    def cmd_Imprime(self, no):
        if isinstance(no.expr, A.Texto):
            self.emit(f'print "{no.expr.valor}"')
        else:
            self.emit(f"print {self.expr(no.expr)}")


def gerar_tac(programa):
    return GeradorTAC().gerar(programa)


In [ ]:
%%writefile main.py
# -*- coding: utf-8 -*-
"""
Ponto de entrada do compilador.

Uso:
    python main.py programa.grr            # AST + código de três endereços
    python main.py programa.grr --so-ast   # só a AST
    python main.py programa.grr --so-tac   # só o TAC
    python main.py programa.grr --debug    # grava build/parser.out (tabela LALR)
    python main.py --tokens programa.grr   # lista os tokens reconhecidos
    python main.py --palavras              # tabela de palavras reservadas
"""

import argparse
import os
import sys

from guerrinha import frases
from guerrinha.parser import analisar
from guerrinha.lexer import criar_lexer
from guerrinha.semantica import verificar
from guerrinha.tac import gerar_tac
from guerrinha.ast_nodes import imprimir_ast


def compilar(codigo, mostrar_ast=True, mostrar_tac=True, debug=False):
    """Roda as fases em sequência. Devolve True se o programa foi aceito."""
    outdir = None
    if debug:
        outdir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "build")
        os.makedirs(outdir, exist_ok=True)

    # Fase 1: léxico + sintático (o parser puxa tokens do lexer)
    ast, erros = analisar(codigo, debug=debug, outputdir=outdir)
    if erros:
        for e in erros:
            print(e)
        return False

    # Fase 2: semântica
    erros = verificar(ast)
    if erros:
        for e in erros:
            print(e)
        return False

    # Fase 3: tradução
    if mostrar_ast:
        print("--- AST ---")
        print(imprimir_ast(ast))
    if mostrar_tac:
        print("--- Código de três endereços ---")
        print("\n".join(gerar_tac(ast)))
    print(f'  >> "{frases.PIADA_SUCESSO}"')
    print(f"  [{frases.NOME_LINGUAGEM}] Programa aceito.")
    return True


def listar_tokens(codigo):
    lexer = criar_lexer()
    lexer.input(codigo)
    for tok in lexer:
        print(f"linha {tok.lineno:>3}  {tok.type:<15} {tok.value!r}")
    for e in lexer.erros:
        print(e)


def tabela_palavras():
    print("| Token | Construção | Frase |")
    print("|---|---|---|")
    papeis = {
        "TIPO_INT": "tipo inteiro", "TIPO_FLOAT": "tipo fracionário",
        "SE": "if", "SENAOSE": "else if", "SENAO": "else",
        "ESCOLHA": "switch", "CASO": "case", "PARE": "break",
        "PADRAO": "default", "ENQUANTO": "while", "IMPRIME": "print",
    }
    for tok, f in frases.PALAVRAS_RESERVADAS.items():
        print(f"| {tok} | {papeis.get(tok, '')} | `{f}` |")


def main():
    ap = argparse.ArgumentParser(description=f"Compilador {frases.NOME_LINGUAGEM}")
    ap.add_argument("arquivo", nargs="?")
    ap.add_argument("--so-ast", action="store_true")
    ap.add_argument("--so-tac", action="store_true")
    ap.add_argument("--debug", action="store_true")
    ap.add_argument("--tokens", action="store_true")
    ap.add_argument("--palavras", action="store_true")
    args = ap.parse_args()

    if args.palavras:
        tabela_palavras()
        return 0
    if not args.arquivo:
        ap.error("informe o arquivo-fonte")

    with open(args.arquivo, encoding="utf-8") as f:
        codigo = f.read()

    if args.tokens:
        listar_tokens(codigo)
        return 0

    print(f"=== {frases.NOME_LINGUAGEM} · {os.path.basename(args.arquivo)} ===")
    ok = compilar(codigo,
                  mostrar_ast=not args.so_tac,
                  mostrar_tac=not args.so_ast,
                  debug=args.debug)
    return 0 if ok else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile ferramentas/rodar_testes.py
# -*- coding: utf-8 -*-
"""
Roda todos os programas de testes/ e grava as saídas em testes/saidas/.

Válidos devem ser ACEITOS; inválidos devem ser REJEITADOS.
Uso (na raiz do projeto):  python ferramentas/rodar_testes.py
"""

import contextlib
import io
import os
import sys

RAIZ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, RAIZ)

from main import compilar  # noqa: E402

TESTES = os.path.join(RAIZ, "testes")
SAIDAS = os.path.join(TESTES, "saidas")


def main():
    os.makedirs(SAIDAS, exist_ok=True)
    falhas = 0
    for pasta, esperado in (("validos", True), ("invalidos", False)):
        for nome in sorted(os.listdir(os.path.join(TESTES, pasta))):
            caminho = os.path.join(TESTES, pasta, nome)
            with open(caminho, encoding="utf-8") as f:
                codigo = f.read()
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                aceito = compilar(codigo)
            saida = os.path.join(SAIDAS, os.path.splitext(nome)[0] + ".txt")
            with open(saida, "w", encoding="utf-8") as f:
                f.write(buf.getvalue())
            ok = aceito == esperado
            falhas += not ok
            status = "ok   " if ok else "FALHA"
            resultado = "aceito" if aceito else "rejeitado"
            print(f"[{status}] {pasta}/{nome:<32} -> {resultado}")
    print(f"\n{falhas} falha(s). Saídas em testes/saidas/")
    return 1 if falhas else 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile ferramentas/experimento_conflitos.py
# -*- coding: utf-8 -*-
"""
Experimento para o relatório (seção 3, Decisões de projeto):
gera a tabela LALR COM e SEM a tabela de precedência dos operadores
aritméticos e mostra quantos conflitos shift/reduce aparecem.

Uso (na raiz do projeto):  python ferramentas/experimento_conflitos.py
Gera build/com_precedencia.out e build/sem_precedencia.out.
"""

import os
import re
import sys

RAIZ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, RAIZ)

import inspect  # noqa: E402
import ply.yacc as yacc  # noqa: E402
import guerrinha.parser as P  # noqa: E402

BUILD = os.path.join(RAIZ, "build")
os.makedirs(BUILD, exist_ok=True)


def gerar(nome_arquivo):
    kw = dict(module=P, debug=True, debugfile=nome_arquivo, outputdir=BUILD)
    if "write_tables" in inspect.signature(yacc.yacc).parameters:
        kw["write_tables"] = False
    yacc.yacc(**kw)
    with open(os.path.join(BUILD, nome_arquivo), encoding="utf-8") as f:
        txt = f.read()
    sr = len(re.findall(r"shift/reduce conflict for \S+ in state", txt))
    rr = len(re.findall(r"reduce/reduce conflict in state", txt))
    return sr, rr


original = P.precedence
print("COM precedência:", "S/R=%d R/R=%d" % gerar("com_precedencia.out"))
P.precedence = (("right", "MENOS_UNARIO"),)    # remove só os binários
print("SEM precedência:", "S/R=%d R/R=%d" % gerar("sem_precedencia.out"))
P.precedence = original
print("Detalhes (estados e tokens de cada conflito) em build/*.out")


## 4. Programas de teste

In [ ]:
%%writefile testes/validos/v1_declaracoes_expressoes.grr
// V1 - Declarações (individual, em lista, com inicialização),
//      precedência/associatividade e menos unário.
// Esperado: ACEITO. No TAC, 10 + 5 * 2 calcula a multiplicação primeiro
//           e 20 - 4 - 1 associa à esquerda ((20 - 4) - 1).
radio limite, faixa;
radio base = 3;
gaucha media, taxa = 2.5;

limite = 10 + 5 * 2;
faixa = 20 - 4 - 1;
base = -(base + 1) * 2;
media = limite / 2;          // radio / radio = radio, depois alarga para gaucha
taxa = taxa * base + 0.5;    // base é convertida para float no TAC

Vou botar uma gravata pra te ouvir (media);


In [ ]:
%%writefile testes/validos/v2_condicional.grr
// V2 - Condicional com os três formatos (se / se-senão / se-senãose-senão)
//      e os seis operadores relacionais.
// Esperado: ACEITO.
radio a = 7, b = 3;
gaucha r;

Ninguém quer saber o preço do dólar (a > b) {
    r = 1.0;
}

Ninguém quer saber o preço do dólar (a == b) {
    r = 0.0;
}, nós não temos dólar {
    r = a - b;
}

Ninguém quer saber o preço do dólar (a < b) {
    r = -1.0;
} Se algo deu certo, então tem que mudar (a >= 10) {
    r = 10.0;
} Se algo deu certo, então tem que mudar (a <= 0) {
    r = 0.0;
} Se algo deu certo, então tem que mudar (a != b) {
    r = 2.0;
}, nós não temos dólar {
    r = 3.0;
}


In [ ]:
%%writefile testes/validos/v3_selecao_multipla.grr
// V3 - Seleção múltipla: caso unitário, caso múltiplo, caso sem
//      encerramento (cai no próximo), caso padrão.
// Esperado: ACEITO. No TAC, o caso 3 não tem "goto" de saída: cai no 4.
radio dia = 3, tipo;

O time vai jogar motivado? (dia) {
    Então vai perder motivado 1:
        tipo = 10;
        Tem gato na tuba;
    Então vai perder motivado 2, 6, 7:
        tipo = 20;
        Tem gato na tuba;
    Então vai perder motivado 3:
        tipo = 30;
    Então vai perder motivado 4:
        tipo = tipo + 1;
        Tem gato na tuba;
    Butuca tira boi do mato:
        tipo = 0;
}

// seleção sem caso padrão
O time vai jogar motivado? (tipo - 30) {
    Então vai perder motivado -1: tipo = 1; Tem gato na tuba;
    Então vai perder motivado 1:  tipo = 2; Tem gato na tuba;
}


In [ ]:
%%writefile testes/validos/v4_programa_completo.grr
// V4 - Programa completo: laço, seleção dentro do laço, escopo de bloco
//      e mistura de radio e gaucha.
// Esperado: ACEITO.
radio i = 0, pares = 0, impares = 0;
gaucha soma = 0.0, media;

Dormindo no gelo e acordando suado (i < 10) {
    radio resto;                     // variável local ao bloco do laço
    resto = i - (i / 2) * 2;
    O time vai jogar motivado? (resto) {
        Então vai perder motivado 0: pares = pares + 1; Tem gato na tuba;
        Butuca tira boi do mato: impares = impares + 1;
    }
    soma = soma + i;
    i = i + 1;
}

Ninguém quer saber o preço do dólar (i != 0) {
    media = soma / i;
}, nós não temos dólar {
    media = 0;
}

Vou botar uma gravata pra te ouvir (media);
Vou botar uma gravata pra te ouvir (pares * 100 + impares);


In [ ]:
%%writefile testes/validos/v5_hello_world.grr
// V5 - Hello World: imprime um texto entre aspas e, depois, um número.
// Esperado: ACEITO.
Vou botar uma gravata pra te ouvir ("Hello, World!");

radio ano = 2026;
Vou botar uma gravata pra te ouvir ("Olá do Little_Guerra, no ano de");
Vou botar uma gravata pra te ouvir (ano);


In [ ]:
%%writefile testes/invalidos/e1_nao_declarada.grr
// E1 - ERRO SEMÂNTICO: uso de variável não declarada, redeclaração
//      e variável usada fora do bloco em que foi declarada.
// Esperado: 4 erros semânticos (linhas 8, 9, 10 e 15).
radio a;
gaucha b;

a = 5;
c = a + 1;          // 'c' não declarada
a = d * 2;          // 'd' não declarada
radio a;            // redeclaração de 'a'

Ninguém quer saber o preço do dólar (a > 0) {
    radio local = 1;
}
a = local;          // 'local' só existe dentro do bloco


In [ ]:
%%writefile testes/invalidos/e2_tipos.grr
// E2 - ERRO SEMÂNTICO DE TIPOS: valor gaucha atribuído a variável radio.
// gaucha <- radio é permitido (alargamento); radio <- gaucha NÃO é.
// Esperado: 3 erros de tipo (linhas 8, 9 e 10).
radio n;
gaucha f = 3;       // ok: radio -> gaucha
f = n;              // ok

n = 2.5;            // erro: literal gaucha em radio
n = f * 2;          // erro: gaucha * radio = gaucha
radio m = n / 2.0;  // erro: inicialização com gaucha


In [ ]:
%%writefile testes/invalidos/e3_selecao.grr
// E3 - ERROS SEMÂNTICOS NA SELEÇÃO MÚLTIPLA E NO ENCERRAMENTO DE CASO.
// Esperado: expressão gaucha na seleção (linha 7), rótulo repetido
//           (linha 9) e "Tem gato na tuba" fora de seleção/laço (linha 13).
radio x = 1;
gaucha y = 1.5;

O time vai jogar motivado? (y) {
    Então vai perder motivado 1: x = 1; Tem gato na tuba;
    Então vai perder motivado 2, 1: x = 2; Tem gato na tuba;
}

x = 3;
Tem gato na tuba;


In [ ]:
%%writefile testes/invalidos/l1_caractere_invalido.grr
// L1 - ERRO LÉXICO: '@' e '$' não pertencem ao alfabeto da linguagem.
// Esperado: erro léxico na linha 5 (e na 6); o programa é rejeitado.
radio a;
a = 2;
a = a @ 3;
radio $b;


In [ ]:
%%writefile testes/invalidos/l2_texto_sem_fechar.grr
// L2 - ERRO LÉXICO: o texto abre aspas e não fecha na mesma linha.
// Esperado: erro léxico na linha 3; o programa é rejeitado.
Vou botar uma gravata pra te ouvir ("Hello, World!);


In [ ]:
%%writefile testes/invalidos/s1_falta_delimitador.grr
// S1 - ERRO SINTÁTICO: falta o ';' no fim da atribuição da linha 6.
// Esperado: erro sintático na linha 7 (o parser só percebe a falta do ';'
//           quando vê o próximo token, 'b', que está na linha 7).
radio a, b;
a = 1;
a = a + 2
b = a;


In [ ]:
%%writefile testes/invalidos/s2_bloco_e_caso.grr
// S2 - ERRO SINTÁTICO: caso padrão antes de um caso comum.
// A gramática exige: casos primeiro, padrão (opcional) por último.
// Esperado: erro sintático na linha 11, na palavra reservada "Então vai perder motivado".
radio x = 2;

O time vai jogar motivado? (x) {
    Então vai perder motivado 1:
        x = 0;
    Butuca tira boi do mato:
        x = 5;
    Então vai perder motivado 2:
        x = 9;
}


In [ ]:
%%writefile testes/invalidos/s3_expressao_malformada.grr
// S3 - ERRO SINTÁTICO: operador sem operando e parêntese não fechado.
// Esperado: erro sintático na linha 6, token inesperado '*'.
radio a = 1;
gaucha f;

f = (a + * 2;


## 5. Execução
### 5.1 Todos os testes

In [ ]:
!python ferramentas/rodar_testes.py

### 5.2 Conflitos da gramática (com e sem precedência)

In [ ]:
!python ferramentas/experimento_conflitos.py

### 5.3 Programas válidos (AST + código de três endereços)

In [ ]:
!python main.py testes/validos/v1_declaracoes_expressoes.grr

In [ ]:
!python main.py testes/validos/v2_condicional.grr

In [ ]:
!python main.py testes/validos/v3_selecao_multipla.grr

In [ ]:
!python main.py testes/validos/v4_programa_completo.grr

In [ ]:
!python main.py testes/validos/v5_hello_world.grr

### 5.4 Programas inválidos (mensagem + linha)

In [ ]:
!python main.py testes/invalidos/e1_nao_declarada.grr

In [ ]:
!python main.py testes/invalidos/e2_tipos.grr

In [ ]:
!python main.py testes/invalidos/e3_selecao.grr

In [ ]:
!python main.py testes/invalidos/l1_caractere_invalido.grr

In [ ]:
!python main.py testes/invalidos/l2_texto_sem_fechar.grr

In [ ]:
!python main.py testes/invalidos/s1_falta_delimitador.grr

In [ ]:
!python main.py testes/invalidos/s2_bloco_e_caso.grr

In [ ]:
!python main.py testes/invalidos/s3_expressao_malformada.grr

## 6. Teste livre
Edite o programa abaixo e execute as duas células.

In [ ]:
%%writefile meu_programa.grr
radio x = 1;
Vou botar uma gravata pra te ouvir (x + 1);


In [ ]:
!python main.py meu_programa.grr